# Figure 6 — analysis (CD8 T cells: Tpex vs Tex)
Computes every Figure 6 result and the supplementary tables and writes them to `data/`; the panels are drawn from these
tables in `cd8_figures.ipynb` (Figure 6 is assembled in Illustrator, `../../illustrator-figures/Figure6.ai`; panel 6a,
the schematic, lives there).

Datasets: Hu et al. 2026 (LCMV clone 13 chronic infection) and Li et al. 2025 (E.G7-OVA tumor), progenitor-exhausted
(Tpex) vs effector-exhausted (Tex) CD8 T cells. Interchromosomal LAS contact graphs (`G_all_filt_20.0.pkl`), the mm10
250 kb bins, the gene-set collections, the set catalog and the Hu Tex-vs-Tpex ranking are read from
`/data/land/3d-inter-paper-data/`. Runs in ~85 s.

In [1]:
import json
import os
import pickle
import sys

import networkx as nx
import numpy as np
import pandas as pd
from scipy.stats import hypergeom

sys.path.insert(0, "..")                                # shared modules
import coloc_stats                                      # contact counting, per-state hypergeometric test + BH
import gsea                                             # preranked GSEA
import kcore                                            # state-specific k-cores

ANNOT = "/data/land/3d-inter-paper-data/cd8-hu/annot/mm10_250kb"
FILT = "20.0"
STATES = ["T_PEX", "T_EX_EFF"]                          # Tpex-specific, Tex-specific edge types (+ "Shared")
GENE_ATTR = "protein_coding_genes_with_tss"             # node attribute: the genes with a TSS in each 250 kb bin
HU_GRAPH = "/data/land/3d-inter-paper-data/cd8-hu/runs/pooled-v2/analysis/G_all_filt_20.0.pkl"     # Hu (LCMV)
LI_GRAPH = "/data/land/3d-inter-paper-data/li-2025/runs/pooled-v2/analysis/G_all_filt_20.0.pkl"    # Li (tumor)
# (dataset, G_all graph [read], output folder [write])
DATASETS = [("cd8-hu", HU_GRAPH, "data/cd8-hu"),
            ("li-2025", LI_GRAPH, "data/li-2025")]

# the five program sets of Figure 6b, c: set id -> predicted pole (clean names come from the catalog)
PROGRAM_SETS = {
    "T_EX_EFF core top100 (ExpC-core)": "T_EX_EFF",               # Effector core (Hu 2026, top-100)
    "Eff-like top100 (T_EX_EFF) (Giles)": "T_EX_EFF",             # Effector-like (Giles 2022, top-100)
    "Day8-effector-vs-memory UP (Kaech) (external)": "T_EX_EFF",  # Effector (vs memory) (Kaech 2002)
    "T_PEX core top100 (ExpC-core)": "T_PEX",                     # Progenitor core (Hu 2026, top-100)
    "ExpC4 top100 (T_PEX) (ExpC-topN)": "T_PEX",                  # Stem/metabolic (Hu 2026, top-100)
}
SETS = list(PROGRAM_SETS)

In [2]:
# inputs: the gene-set collections, the set catalog and the two cell-state-specific contact graphs
with open(f"{ANNOT}/mm10_pathways_final.json") as fh:
    pw_final = json.load(fh)                            # 77 finalized sets (60 unbiased + 17 CD8-specific); BH over these
catalog = pd.read_csv(f"{ANNOT}/pathway_catalog.csv").set_index("set_id")   # clean name, tier, pole per final set
graphs = {}
for ds, path, _ in DATASETS:
    with open(path, "rb") as fh:
        graphs[ds] = pickle.load(fh)
print(f"{len(pw_final)} finalized gene sets")

77 finalized gene sets


In [3]:
# the five program sets (ids, clean names, poles) in their fixed order, for the figures notebook
os.makedirs("data", exist_ok=True)
program_sets = pd.DataFrame({"set_id": SETS,
                             "clean": [catalog.loc[s, "clean"] for s in SETS],
                             "pole": [PROGRAM_SETS[s] for s in SETS]})
program_sets.to_csv("data/program_sets.csv", index=False)
program_sets

,set_id,clean,pole
0,T_EX_EFF core top100 (ExpC-core),"Effector core (Hu 2026, top-100)",T_EX_EFF
1,Eff-like top100 (T_EX_EFF) (Giles),"Effector-like (Giles 2022, top-100)",T_EX_EFF
2,Day8-effector-vs-memory UP (Kaech) (external),Effector (vs memory) (Kaech 2002),T_EX_EFF
3,T_PEX core top100 (ExpC-core),"Progenitor core (Hu 2026, top-100)",T_PEX
4,ExpC4 top100 (T_PEX) (ExpC-topN),"Stem/metabolic (Hu 2026, top-100)",T_PEX


## Pathway colocalization (Figure 6c)
Per gene set and dataset (shared `../coloc_stats.py`): unique gene–gene contacts per state, per-state hypergeometric
test against the genome-wide
background (BH over the 77 finalized gene sets of `mm10_pathways_final.json`), and the complementary Fisher test of Tex-
vs Tpex-specific contacts against the genome (supplementary table; not plotted). Writes
`data/<dataset>/pathway_enrichment_filt_20.0.csv` and `differential_specificity_filt_20.0.csv`.

In [4]:
def enrichment_label(r):
    """Which states a set is enriched in (FDR < 0.05)."""
    pex_sig = r["fdr_T_PEX"] < 0.05
    eff_sig = r["fdr_T_EX_EFF"] < 0.05
    if pex_sig and eff_sig:
        return "Both"
    if pex_sig:
        return "T_PEX only"
    if eff_sig:
        return "T_EX_EFF only"
    return "None"


def enrichment_table(G, bg):
    """Per set: state-specific contact counts, shares, hypergeometric p and BH FDR (over all sets), and a label."""
    M = sum(bg.get(s, 0) for s in STATES) + bg.get("Shared", 0)          # population: all genome gene-gene contacts
    rows = []
    for name, genes in pw_final.items():
        counts = coloc_stats.count_contacts(G, GENE_ATTR, genes)
        N = sum(counts.get(s, 0) for s in STATES) + counts.get("Shared", 0)   # draws: the set's contacts
        rec = {"pathway": name, "n_pathway_contacts": N}
        for s in STATES:
            k = counts.get(s, 0)                                          # successes: the set's state-specific contacts
            rec[f"n_{s}"] = k
            rec[f"pct_{s}"] = 100 * k / N if N else np.nan
            rec[f"pval_{s}"] = coloc_stats.hypergeom_enrichment_pval(k, M, bg.get(s, 0), N) if N else np.nan
        rows.append(rec)
    res = pd.DataFrame(rows)
    for s in STATES:
        res[f"fdr_{s}"] = coloc_stats.bh(res[f"pval_{s}"])
        res[f"nlog10_fdr_{s}"] = -np.log10(res[f"fdr_{s}"])
    res["enrichment"] = res.apply(enrichment_label, axis=1)
    return res


def specificity_table(res, bg):
    """Fisher test of Tex- vs Tpex-specific contacts per set against the genome; fisher_or = Tex fold / Tpex fold."""
    spec = res[["pathway", "n_pathway_contacts", "n_T_PEX", "n_T_EX_EFF", "fdr_T_PEX", "fdr_T_EX_EFF"]].copy()
    fisher = [coloc_stats.differential_fisher(tex_n, tpex_n, bg["T_EX_EFF"], bg["T_PEX"])
              for tex_n, tpex_n in zip(spec.n_T_EX_EFF, spec.n_T_PEX)]
    spec["fisher_or"] = [f[0] for f in fisher]
    spec["fisher_p"] = [f[1] for f in fisher]
    spec["fisher_fdr"] = coloc_stats.bh(spec.fisher_p)
    return spec

In [5]:
genome_contacts = {}                                    # genome-wide unique gene-gene contacts per state, per dataset
enrichment = {}
for ds, _, out_dir in DATASETS:
    os.makedirs(out_dir, exist_ok=True)
    genome_contacts[ds] = coloc_stats.count_contacts(graphs[ds], GENE_ATTR)
    enrichment[ds] = enrichment_table(graphs[ds], genome_contacts[ds])
    res_sorted = enrichment[ds].sort_values("nlog10_fdr_T_PEX", ascending=False)
    res_sorted.to_csv(f"{out_dir}/pathway_enrichment_filt_{FILT}.csv", index=False)
    spec = specificity_table(enrichment[ds], genome_contacts[ds])   # unsorted input: keeps the row order of ties
    spec.sort_values("fisher_p").to_csv(f"{out_dir}/differential_specificity_filt_{FILT}.csv", index=False)
    n_enriched = (enrichment[ds].enrichment != "None").sum()
    print(f"{ds}: {n_enriched} sets enriched in at least one state (FDR < 0.05)")

cd8-hu: 18 sets enriched in at least one state (FDR < 0.05)


li-2025: 20 sets enriched in at least one state (FDR < 0.05)


## Contact composition of the five program sets (Figure 6c background; supplementary table)
The five program sets (+ the genome background) split into Tpex-specific / Tex-specific / Shared gene–gene contacts
(# and %), per dataset. The genome row gives the background shares of the Figure 6c fold enrichment. Writes
`data/<dataset>/contact_composition_5sets.csv`.

In [6]:
CATS = STATES + ["Shared"]                              # column order: Tpex, Tex, Shared


def composition_row(name, counts):
    """One table row: total contacts and the Tpex-only / Tex-only / Shared counts and percentages."""
    N = sum(counts.get(c, 0) for c in CATS)
    r = {"set": name, "N": N}
    for c, lab in zip(CATS, ["Tpex_only", "Tex_only", "Shared"]):
        r[f"{lab}_n"] = counts.get(c, 0)
        r[f"{lab}_%"] = round(100 * counts.get(c, 0) / (N or 1), 1)   # (N or 1): avoid /0 for a set without contacts
    return r

In [7]:
for ds, _, out_dir in DATASETS:
    rows = [composition_row("Background (genome)", genome_contacts[ds])]
    for sid in SETS:
        counts = coloc_stats.count_contacts(graphs[ds], GENE_ATTR, pw_final[sid])
        rows.append(composition_row(catalog.loc[sid, "clean"], counts))
    composition = pd.DataFrame(rows)
    composition.to_csv(f"{out_dir}/contact_composition_5sets.csv", index=False)
    print(f"\n{ds}")
    print(composition.to_string(index=False))


cd8-hu
                                set        N  Tpex_only_n  Tpex_only_%  Tex_only_n  Tex_only_%  Shared_n  Shared_%
                Background (genome) 27213378      5237856         19.2     5249790        19.3  16725732      61.5
   Effector core (Hu 2026, top-100)      846          128         15.1         299        35.3       419      49.5
Effector-like (Giles 2022, top-100)     1160          181         15.6         288        24.8       691      59.6
  Effector (vs memory) (Kaech 2002)     3610          642         17.8         838        23.2      2130      59.0
 Progenitor core (Hu 2026, top-100)      938          211         22.5         167        17.8       560      59.7
  Stem/metabolic (Hu 2026, top-100)      856          159         18.6         168        19.6       529      61.8

li-2025
                                set        N  Tpex_only_n  Tpex_only_%  Tex_only_n  Tex_only_%  Shared_n  Shared_%
                Background (genome) 30049320      3901709      

## Pathway expression (Figure 6b)
Preranked GSEA (shared `../gsea.py`) of the 17 CD8-specific finalized sets (FDR over these 17). The ranking is the Hu
bulk RNA-seq Tex-vs-Tpex signal-to-noise (`hu_tex_vs_tpex_s2n.rnk`) over the ~3,800-gene DEG list published as a
supplementary table — the only Hu expression data available — rather than the DESeq2 Wald statistic of the other
figures; gseapy's "Duplicated values" warning is expected (ties for 0.05% of genes). Writes
`data/cd8-hu/gsea_hu_tex_vs_tpex.csv` (sets by clean name).

In [8]:
rnk = pd.read_csv(f"{ANNOT}/hu_tex_vs_tpex_s2n.rnk", sep="\t", header=None, names=["gene", "score"])
gs_cd8 = {}                                             # the 17 CD8-specific sets, keyed by clean name
for set_id, genes in pw_final.items():
    if catalog.loc[set_id, "tier"] != "unbiased public pathway":
        gs_cd8[catalog.loc[set_id, "clean"]] = genes
gsea_hu = gsea.prerank(rnk, gs_cd8)
gsea_hu.to_csv("data/cd8-hu/gsea_hu_tex_vs_tpex.csv", index=False)
program_names = [catalog.loc[s, "clean"] for s in SETS]
gsea_hu.set_index("Term").loc[program_names, ["NES", "NOM p-val", "FDR q-val"]]

2026-10-21 01:50:26,661 [WARNING] Duplicated values found in preranked stats: 0.05% of genes
The order of those genes will be arbitrary, which may produce unexpected results.


,NES,NOM p-val,FDR q-val
Term,,,
"Effector core (Hu 2026, top-100)",3.035547,0.0,0.0
"Effector-like (Giles 2022, top-100)",2.726040,0.0,0.0
Effector (vs memory) (Kaech 2002),2.765552,0.0,0.0
"Progenitor core (Hu 2026, top-100)",-3.112925,0.0,0.0
"Stem/metabolic (Hu 2026, top-100)",-2.897330,0.0,0.0


## CD8-specific gene sets (supplementary table)
All 17 CD8-specific finalized sets (5 shown in Figure 6c): per dataset, the fold enrichment of Tex- and Tpex-specific
contacts vs the genome-wide share and the per-state hypergeometric FDR (BH over the 77 finalized gene sets, as 6c), and
the Hu GSEA NES and FDR q (as 6b). Writes `data/cd8_specific_sets_colocalization.csv`.

In [9]:
cd8_ids = [s for s in pw_final if catalog.loc[s, "tier"] != "unbiased public pathway"]
gsea_by_name = gsea_hu.set_index("Term")
table = pd.DataFrame({"set_id": cd8_ids,
                      "clean": [catalog.loc[s, "clean"] for s in cd8_ids],
                      "tier": [catalog.loc[s, "tier"] for s in cd8_ids],
                      "source": [catalog.loc[s, "source"] for s in cd8_ids],
                      "pole": [catalog.loc[s, "pole"] for s in cd8_ids],
                      "n_genes": [len(pw_final[s]) for s in cd8_ids],
                      "in_fig6c": [s in PROGRAM_SETS for s in cd8_ids]})
for ds, key in (("cd8-hu", "hu"), ("li-2025", "li")):
    bg = genome_contacts[ds]
    bg_total = sum(bg.get(c, 0) for c in CATS)
    e = enrichment[ds].set_index("pathway").loc[cd8_ids]
    for state, short in (("T_EX_EFF", "tex"), ("T_PEX", "tpex")):
        genome_pct = 100 * bg[state] / bg_total         # the genome-wide share (Figure 6c background)
        table[f"{key}_fold_{short}"] = (e[f"pct_{state}"] / genome_pct).values
        table[f"{key}_fdr_{short}"] = e[f"fdr_{state}"].values
table["hu_gsea_nes"] = gsea_by_name.loc[table.clean, "NES"].values
table["hu_gsea_fdr"] = gsea_by_name.loc[table.clean, "FDR q-val"].values
table.to_csv("data/cd8_specific_sets_colocalization.csv", index=False)
table[["clean", "pole", "in_fig6c", "hu_fold_tex", "hu_fdr_tex", "li_fold_tex", "li_fdr_tex", "hu_gsea_nes"]]

,clean,pole,in_fig6c,hu_fold_tex,hu_fdr_tex,li_fold_tex,li_fdr_tex,hu_gsea_nes
0,Stemness/precursor markers (curated),T_PEX,False,0.431976,1.000000e+00,1.016238,1.000000e+00,-2.097362
1,Exhaustion markers (curated),shared,False,2.159878,2.364574e-01,2.830948,6.730629e-04,-0.784888
2,Effector (vs exhausted) (Wherry 2007),T_EX_EFF,False,0.762228,1.000000e+00,0.742668,1.000000e+00,2.207615
3,Exhausted (vs effector) (Wherry 2007),T_EX_EFF,False,1.276269,3.758122e-07,1.195077,3.595774e-05,-1.675516
4,Exhausted (vs memory) (Wherry 2007),T_EX_EFF,False,1.085005,7.834671e-02,1.032133,5.468577e-01,1.149263
5,Memory (vs exhausted) (Wherry 2007),T_PEX,False,0.841688,1.000000e+00,0.856977,1.000000e+00,1.611549
6,Effector (vs memory) (Kaech 2002),T_EX_EFF,True,1.203309,4.643831e-08,1.278938,7.859396e-19,2.765552
7,"Shared exhaustion/activation (Hu 2026, top-100)",T_PEX,False,0.823745,1.000000e+00,1.013666,9.832255e-01,-0.587829
8,"Stem/metabolic (Hu 2026, top-100)",T_PEX,True,1.017363,1.000000e+00,0.894622,1.000000e+00,-2.897330
9,"Effector core (Hu 2026, top-100)",T_EX_EFF,True,1.832067,4.458572e-26,1.562657,7.422214e-18,3.035547


In [10]:
# how many sets colocalize in their predicted state, in both / one / neither dataset
for pole, state in (("T_EX_EFF", "tex"), ("T_PEX", "tpex"), ("shared", None)):
    sub = table[table.pole == pole]
    if state is None:
        print(f"shared pole: {len(sub)} set ({', '.join(sub.clean)})")
        continue
    n_sig = (sub[f"hu_fdr_{state}"] < 0.05).astype(int) + (sub[f"li_fdr_{state}"] < 0.05).astype(int)
    print(f"{pole} pole ({len(sub)} sets), {state}-specific FDR < 0.05: both {(n_sig == 2).sum()}, "
          f"one {(n_sig == 1).sum()}, neither {(n_sig == 0).sum()}")

T_EX_EFF pole (9 sets), tex-specific FDR < 0.05: both 4, one 1, neither 4
T_PEX pole (7 sets), tpex-specific FDR < 0.05: both 0, one 0, neither 7
shared pole: 1 set (Exhaustion markers (curated))


## Degree-preserving null (supplementary table)
Per program set, on its predicted-pole layer (Tex-specific for effector, Tpex-specific for progenitor sets): a
configuration-model null that keeps each bin's degree in that layer and rewires partners (1,000 rewirings);
fold = observed within-set gene–gene contacts / null mean, empirical p = (# rewirings ≥ observed + 1) / 1,001 (minimum
0.001). One RNG (seed 0) consumed Hu then Li. Writes `data/<dataset>/degree_null_5sets.csv`.

In [11]:
NPERM = 1000


def sample_distinct(T, m):
    """m distinct positions in [0, T) via rejection (fast for m << T); draws from the shared `rng` (seeded below)."""
    s = np.unique(rng.integers(0, T, size=int(m * 1.3) + 8))
    while len(s) < m:
        s = np.unique(np.concatenate([s, rng.integers(0, T, size=m)]))
    return s[:m]


def degree_null(G):
    """Observed within-set contacts vs the degree-preserving null for the five program sets of one graph."""
    nodes = list(G.nodes())
    idx = {n: i for i, n in enumerate(nodes)}
    node_genes = [set(G.nodes[n].get("protein_coding_genes_with_tss", []) or []) for n in nodes]

    # edges of each state-specific layer as (node index, node index), then as two arrays of edge ends
    layer_edges = {"T_PEX": [], "T_EX_EFF": []}
    for (a, b), t in nx.get_edge_attributes(G, "edge_type").items():
        if t in layer_edges:
            layer_edges[t].append((idx[a], idx[b]))
    layer_ends = {}
    for t, edges in layer_edges.items():
        layer_ends[t] = (np.array([e[0] for e in edges]), np.array([e[1] for e in edges]))

    rows = []
    for sid, pole in PROGRAM_SETS.items():
        name = catalog.loc[sid, "clean"]
        ps = set(pw_final[sid])
        w = np.array([len(g & ps) for g in node_genes], dtype=float)    # set genes per bin
        u, v = layer_ends[pole]
        obs = float((w[u] * w[v]).sum())                # within-set gene-gene contacts

        stub_weights = np.concatenate([w[u], w[v]])
        T = len(stub_weights)
        set_stub_weights = stub_weights[stub_weights > 0]
        m = len(set_stub_weights)
        is_pos = np.zeros(T, bool)
        wpos = np.zeros(T)
        nulls = np.empty(NPERM)
        for k in range(NPERM):
            # slots 2i and 2i+1 form one edge; dropping the set's weighted stubs into random slots is a random stub
            # matching with every bin's layer degree fixed (configuration model)
            pos = sample_distinct(T, m)
            shuffled = rng.permutation(set_stub_weights)
            is_pos[pos] = True
            wpos[pos] = shuffled
            partner = pos ^ 1                           # the other end of each slot's edge
            nulls[k] = (wpos[pos] * wpos[partner] * is_pos[partner]).sum() / 2
            is_pos[pos] = False
            wpos[pos] = 0.0

        null_mean = nulls.mean()
        null_sd = nulls.std() + 1e-9
        rows.append(dict(set=name, pole=pole, observed=round(obs), null_mean=round(null_mean, 1),
                         fold=round(obs / (null_mean + 1e-9), 2),
                         emp_p=round((np.sum(nulls >= obs) + 1) / (NPERM + 1), 4),
                         z=round((obs - null_mean) / null_sd, 1)))
    return pd.DataFrame(rows)

In [12]:
rng = np.random.default_rng(0)                          # one shared stream, consumed Hu then Li: the order fixes the results
for ds, _, out_dir in DATASETS:
    null_table = degree_null(graphs[ds])
    null_table.to_csv(f"{out_dir}/degree_null_5sets.csv", index=False)
    print(f"\n{ds} (fold = observed / degree-matched expectation)")
    print(null_table.to_string(index=False))


cd8-hu (fold = observed / degree-matched expectation)
                                set     pole  observed  null_mean  fold  emp_p    z
   Effector core (Hu 2026, top-100) T_EX_EFF       299      171.4  1.74 0.0010  7.7
Effector-like (Giles 2022, top-100) T_EX_EFF       288      180.3  1.60 0.0010  6.5
  Effector (vs memory) (Kaech 2002) T_EX_EFF       838      496.2  1.69 0.0010 11.5
 Progenitor core (Hu 2026, top-100)    T_PEX       211      168.4  1.25 0.0010  3.2
  Stem/metabolic (Hu 2026, top-100)    T_PEX       159      134.9  1.18 0.0629  1.7



li-2025 (fold = observed / degree-matched expectation)
                                set     pole  observed  null_mean  fold  emp_p    z
   Effector core (Hu 2026, top-100) T_EX_EFF       340      220.3  1.54 0.0010  6.3
Effector-like (Giles 2022, top-100) T_EX_EFF       324      196.5  1.65 0.0010  7.3
  Effector (vs memory) (Kaech 2002) T_EX_EFF      1143      622.2  1.84 0.0010 15.2
 Progenitor core (Hu 2026, top-100)    T_PEX       128       99.8  1.28 0.0030  2.7
  Stem/metabolic (Hu 2026, top-100)    T_PEX       119       98.8  1.20 0.0529  1.7


## Gene overlap among the five program sets (QC; not a figure panel)
Pairwise shared genes (diagonal = set size): each set is largely unique, so agreement across sets is not driven by shared
genes. Writes `data/overlap_5sets.csv`.

In [13]:
overlap = pd.DataFrame(0, index=program_names, columns=program_names)
for a, name_a in zip(SETS, program_names):
    for b, name_b in zip(SETS, program_names):
        overlap.loc[name_a, name_b] = len(set(pw_final[a]) & set(pw_final[b]))   # diagonal = set size
overlap.to_csv("data/overlap_5sets.csv")
overlap

,"Effector core (Hu 2026, top-100)","Effector-like (Giles 2022, top-100)",Effector (vs memory) (Kaech 2002),"Progenitor core (Hu 2026, top-100)","Stem/metabolic (Hu 2026, top-100)"
"Effector core (Hu 2026, top-100)",100,10,15,0,0
"Effector-like (Giles 2022, top-100)",10,100,14,0,0
Effector (vs memory) (Kaech 2002),15,14,199,0,0
"Progenitor core (Hu 2026, top-100)",0,0,0,100,18
"Stem/metabolic (Hu 2026, top-100)",0,0,0,18,100


## Effector k-cores (Figure 6d)
For each dataset independently: the densest k-core of the Tex-specific colocalization graph among the effector-core
genes (Hu 2026 top-100; shared `../kcore.py`, each gene on its first TSS bin), its gene × gene contact-state matrix, and
the overlap of the Hu and Li cores vs chance (one-sided hypergeometric, per gene and per 250 kb bin; genes sharing a
bin, e.g. Ccl3/4/5, enter a core together, so the bin test is the conservative one). Writes
`data/kcores/{hu,li}_effector_{genes,states}.csv`, `kcore_summary.csv` and `core_overlap.csv`.

In [14]:
bins = pd.read_csv(f"{ANNOT}/mm10_250kb_bins_annotated.csv")
gene_to_bin = {}                                        # gene -> its first 250 kb TSS bin
for bin_id, gene_str in bins["protein_coding_genes_with_tss"].items():
    if pd.notna(gene_str):
        for gene in str(gene_str).split(","):
            gene_to_bin.setdefault(gene, bin_id)
gene_bins = {g: {b} for g, b in gene_to_bin.items()}    # kcore API takes a set of bins per gene
bin_chrom = {i: int(c.removeprefix("chr")) for i, c in bins["chrom"].items()}   # mm10 bins: chr1-19 only

eff_core = [g for g in pw_final["T_EX_EFF core top100 (ExpC-core)"] if g in gene_to_bin]
STATE_CODE = {"T_EX_EFF": 1, "T_PEX": 2, "Shared": 3}  # heatmap codes; 0 = no contact
CORE_DATASETS = {"hu": "cd8-hu", "li": "li-2025"}

In [15]:
os.makedirs("data/kcores", exist_ok=True)
cores = {}
summary = []
for key, ds in CORE_DATASETS.items():
    edge_type = nx.get_edge_attributes(graphs[ds], "edge_type")
    module, k = kcore.state_kcore(edge_type, gene_bins, eff_core, "T_EX_EFF", bin_chrom)
    mat = kcore.state_matrix(edge_type, gene_bins, module, STATE_CODE)
    chroms = [bins.loc[gene_to_bin[g], "chrom"] for g in module]
    pd.DataFrame({"gene": module, "chrom": chroms}).to_csv(f"data/kcores/{key}_effector_genes.csv", index=False)
    pd.DataFrame(mat, index=module, columns=module).to_csv(f"data/kcores/{key}_effector_states.csv")
    cores[key] = module
    summary_row = {"dataset": ds, "k": k, "n_genes": len(module), "n_chromosomes": len(set(chroms))}
    for s, c in STATE_CODE.items():
        summary_row[f"pairs_{s}"] = int((mat == c).sum() // 2)
    summary.append(summary_row)
    print(summary_row)
pd.DataFrame(summary).to_csv("data/kcores/kcore_summary.csv", index=False)

{'dataset': 'cd8-hu', 'k': 6, 'n_genes': 35, 'n_chromosomes': 11, 'pairs_T_EX_EFF': 171, 'pairs_T_PEX': 24, 'pairs_Shared': 137}


{'dataset': 'li-2025', 'k': 6, 'n_genes': 40, 'n_chromosomes': 11, 'pairs_T_EX_EFF': 186, 'pairs_T_PEX': 21, 'pairs_Shared': 213}


In [16]:
def module_overlap(universe, a, b):
    """(shared, expected, p) for the overlap of sets a and b drawn from universe."""
    shared = len(a & b)
    expected = len(a) * len(b) / len(universe)
    p = hypergeom.sf(shared - 1, len(universe), len(a), len(b))
    return shared, expected, p


hu_genes = set(cores["hu"])
li_genes = set(cores["li"])
universe_genes = set(eff_core)
hu_bins = {gene_to_bin[g] for g in hu_genes}
li_bins = {gene_to_bin[g] for g in li_genes}
universe_bins = {gene_to_bin[g] for g in universe_genes}
overlap_rows = []
for unit, universe, a, b in [("genes", universe_genes, hu_genes, li_genes),
                             ("bins", universe_bins, hu_bins, li_bins)]:
    shared, expected, p = module_overlap(universe, a, b)
    overlap_rows.append({"unit": unit, "universe": len(universe), "hu": len(a), "li": len(b), "shared": shared,
                         "expected": expected, "p": p})
core_overlap = pd.DataFrame(overlap_rows)
core_overlap.to_csv("data/kcores/core_overlap.csv", index=False)
shared_core = hu_genes & li_genes
print(core_overlap.to_string(index=False))
print(f"shared genes span {len({gene_to_bin[g] for g in shared_core})} bins on "
      f"{len({bins.loc[gene_to_bin[g], 'chrom'] for g in shared_core})} chromosomes")

 unit  universe  hu  li  shared  expected            p
genes       100  35  40      27      14.0 3.019599e-08
 bins        90  27  33      20       9.9 2.401342e-06
shared genes span 20 bins on 10 chromosomes
